# 03 · MP-SENet — Denoising

Notebook de inferencia con MP-SENet (paquete pip de inferencia `MPSENet`, mantenido por
JacobLinCool a partir del repo oficial `yxlu-0102/MP-SENet`). Categoría: **denoising**, igual que
DeepFilterNet — así que además de comparar contra el baseline clásico, esta comparativa te sirve
para contrastar dos modelos de IA distintos en la misma categoría.

Requiere haber ejecutado antes `00_Setup_Base.ipynb` (Drive montado, HF_HOME configurado, utils
guardadas en `utils/`, audio del tutor subido a `audio_samples/`).

**Igual que HTDemucs, `MPSENet` es un paquete pip puro** (sin dependencias nativas que compilar),
así que no hace falta instalar ningún compilador ni reiniciar el kernel a mitad de notebook.


## 1. Montar Drive y configurar entorno

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/Proyecto_Audio'
os.environ['HF_HOME'] = f'{PROJECT_ROOT}/cache'
os.environ['HF_HUB_CACHE'] = f'{PROJECT_ROOT}/cache'


Mounted at /content/drive


## 2. Instalar dependencias comunes

Los paquetes de `pip` no persisten entre sesiones de Colab (solo los archivos en Drive sí), así
que hay que reinstalar estas dependencias en cada sesión nueva.


In [ ]:
!pip install -q librosa soundfile scipy pesq pystoi speechmos onnxruntime matplotlib pandas


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 67.5 MB/s eta 0:00:00


## 3. Importar utilidades comunes (desde utils/ en Drive)

In [ ]:
import sys
sys.path.append(f'{PROJECT_ROOT}/utils')

from audio_utils_funcionescomunes import cargar_audio, guardar_audio, resamplear, normalizar_pico
from audio_utils_memoria_GPU import liberar_memoria_gpu
from audio_utils_metricas_no_intrusivas import calcular_dnsmos
from audio_utils_baselines_clasicos import baseline_denoising_spectral_gating


## 4. Instalar dependencias específicas de MP-SENet

`MPSENet` es un paquete de inferencia pip normal (sin extensiones nativas que compilar), así que
la instalación es directa. Descarga el checkpoint preentrenado desde Hugging Face la primera vez
que se llama a `.from_pretrained(...)` (queda cacheado en `HF_HOME`, que ya apunta a Drive).


In [ ]:
!pip install -q MPSENet


## 5. Cargar el audio de prueba

Muestra los audios disponibles en `audio_samples/` y pide cuál usar.


In [ ]:
carpeta_audios = f'{PROJECT_ROOT}/audio_samples'

print('Audios disponibles en audio_samples/:')
for archivo in os.listdir(carpeta_audios):
    print(f'  - {archivo}')

nombre_audio = input('\nIntroduce el nombre del archivo de audio a usar (con extensión, ej. AUDIO_TFG.wav): ').strip()
RUTA_AUDIO_ORIGINAL = f'{carpeta_audios}/{nombre_audio}'

if not os.path.isfile(RUTA_AUDIO_ORIGINAL):
    raise FileNotFoundError(f'No se ha encontrado el archivo: {RUTA_AUDIO_ORIGINAL}')

# Nombre base sin extensión, para usarlo luego al nombrar los archivos de salida
NOMBRE_BASE = os.path.splitext(nombre_audio)[0]

# Cargamos con su sample rate original; lo resampleamos al del modelo aparte, más abajo
audio_original, sr_original = cargar_audio(RUTA_AUDIO_ORIGINAL, sr_objetivo=None, forzar_mono=True)
print(f'\nAudio cargado: {len(audio_original)/sr_original:.1f} s, {sr_original} Hz')


Audios disponibles en audio_samples/:
  - AUDIO_REVERB_ALBIOL_TFG.wav

Introduce el nombre del archivo de audio a usar (con extensión, ej. AUDIO_TFG.wav): AUDIO_REVERB_ALBIOL_TFG.wav

Audio cargado: 88.0 s, 44100 Hz


## 6. Cargar el modelo MP-SENet

Usamos el checkpoint `MP-SENet-VB` (entrenado en VoiceBank+DEMAND, el estándar habitual de
denoising de voz). El otro checkpoint oficial disponible es `MP-SENet-DNS` (entrenado en el
dataset de Microsoft DNS Challenge) — si más adelante quieres comparar ambos, basta con cambiar
el string del repo de Hugging Face abajo.

Por defecto, el modelo trocea el audio en segmentos de 2 s para no saturar memoria; para un audio
largo como el del tutor esto es justo lo que interesa, así que no tocamos `segment_size`.


In [ ]:
import torch
from MPSENet import MPSENet

dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'

modelo_mpsenet = MPSENet.from_pretrained('JacobLinCool/MP-SENet-VB').to(dispositivo)
sr_modelo = modelo_mpsenet.sampling_rate
print(f'Modelo cargado en {dispositivo}. Sample rate esperado por el modelo: {sr_modelo} Hz')


config.json:   0%|          | 0.00/248 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

Modelo cargado en cuda. Sample rate esperado por el modelo: 16000 Hz


## 7. Inferencia sobre el audio

In [ ]:
# Resampleamos al sample rate que espera el modelo (normalmente 16 kHz)
audio_entrada, _ = cargar_audio(RUTA_AUDIO_ORIGINAL, sr_objetivo=sr_modelo, forzar_mono=True)

audio_mejorado, sr_salida, _ = modelo_mpsenet(audio_entrada)

RUTA_SALIDA = f'{PROJECT_ROOT}/outputs/mpsenet/{NOMBRE_BASE}_denoised.wav'
os.makedirs(os.path.dirname(RUTA_SALIDA), exist_ok=True)
guardar_audio(RUTA_SALIDA, audio_mejorado, sr_salida)
print(f'Audio mejorado guardado en: {RUTA_SALIDA}')


Audio guardado en: /content/drive/MyDrive/Proyecto_Audio/outputs/mpsenet/AUDIO_REVERB_ALBIOL_TFG_denoised.wav
Audio mejorado guardado en: /content/drive/MyDrive/Proyecto_Audio/outputs/mpsenet/AUDIO_REVERB_ALBIOL_TFG_denoised.wav


## 8. Baseline clásico (spectral gating) sobre el mismo audio

In [ ]:
audio_baseline = baseline_denoising_spectral_gating(audio_original, sr_original)

RUTA_BASELINE = f'{PROJECT_ROOT}/outputs/baseline_denoising/{NOMBRE_BASE}_baseline.wav'
os.makedirs(os.path.dirname(RUTA_BASELINE), exist_ok=True)
guardar_audio(RUTA_BASELINE, audio_baseline, sr_original)
print(f'Audio baseline guardado en: {RUTA_BASELINE}')


Audio guardado en: /content/drive/MyDrive/Proyecto_Audio/outputs/baseline_denoising/AUDIO_REVERB_ALBIOL_TFG_baseline.wav
Audio baseline guardado en: /content/drive/MyDrive/Proyecto_Audio/outputs/baseline_denoising/AUDIO_REVERB_ALBIOL_TFG_baseline.wav


## 9. Calcular métricas (DNSMOS)

Como no hay audio limpio de referencia para la grabación del tutor, usamos DNSMOS (métrica
no-intrusiva) sobre: audio original, salida de MP-SENet, y salida del baseline. `calcular_dnsmos`
resamplea a 16 kHz y reduce a mono automáticamente si hace falta.


In [ ]:
import numpy as np

audio_mejorado_arr = np.array(audio_mejorado).squeeze()

dnsmos_original = calcular_dnsmos(audio_original, sr_original)
dnsmos_mpsenet = calcular_dnsmos(audio_mejorado_arr, sr_salida)
dnsmos_baseline = calcular_dnsmos(audio_baseline, sr_original)

print('DNSMOS — Audio original:   ', dnsmos_original)
print('DNSMOS — MP-SENet:         ', dnsmos_mpsenet)
print('DNSMOS — Baseline (no-IA): ', dnsmos_baseline)


DNSMOS — Audio original:    {'ovrl_mos': np.float64(1.3853487473266228), 'sig_mos': np.float64(1.5779884955663657), 'bak_mos': np.float64(1.790054065636629), 'p808_mos': np.float32(2.426125)}
DNSMOS — MP-SENet:          {'ovrl_mos': np.float64(1.789299124796392), 'sig_mos': np.float64(2.297955951810248), 'bak_mos': np.float64(3.333450215042906), 'p808_mos': np.float32(2.258202)}
DNSMOS — Baseline (no-IA):  {'ovrl_mos': np.float64(1.4003822076714478), 'sig_mos': np.float64(1.5782148367336726), 'bak_mos': np.float64(1.8430766797139668), 'p808_mos': np.float32(2.431142)}


## 10. Tabla resumen de la comparativa

In [ ]:
import pandas as pd

resumen = pd.DataFrame([
    {'Version': 'Original (degradado)', 'OVRL': dnsmos_original.get('ovrl_mos'), 'SIG': dnsmos_original.get('sig_mos'), 'BAK': dnsmos_original.get('bak_mos')},
    {'Version': 'MP-SENet (IA)',        'OVRL': dnsmos_mpsenet.get('ovrl_mos'),  'SIG': dnsmos_mpsenet.get('sig_mos'),  'BAK': dnsmos_mpsenet.get('bak_mos')},
    {'Version': 'Baseline (no-IA)',     'OVRL': dnsmos_baseline.get('ovrl_mos'), 'SIG': dnsmos_baseline.get('sig_mos'), 'BAK': dnsmos_baseline.get('bak_mos')},
])
resumen


,Version,OVRL,SIG,BAK
0,Original (degradado),1.385349,1.577988,1.790054
1,MP-SENet (IA),1.789299,2.297956,3.333450
2,Baseline (no-IA),1.400382,1.578215,1.843077


## 11. (Opcional) Comparar contra DeepFilterNet

Si en la misma sesión ya has cargado y guardado la salida de `01_DeepFilterNet_3.ipynb` para
este mismo audio, puedes cargar ese .wav aquí y calcular su DNSMOS para tener los dos modelos
de denoising uno al lado del otro en la misma tabla. Se deja como celda aparte para no obligar
a tener DeepFilterNet cargado en esta misma sesión (evita conflictos de versión de numpy entre
ambos notebooks).


In [16]:
RUTA_DFN = f'{PROJECT_ROOT}/outputs/deepfilternet/{NOMBRE_BASE}_denoised.wav'

if os.path.isfile(RUTA_DFN):
    audio_dfn, sr_dfn = cargar_audio(RUTA_DFN, sr_objetivo=None, forzar_mono=True)
    dnsmos_dfn = calcular_dnsmos(audio_dfn, sr_dfn)
    print('DNSMOS — DeepFilterNet (IA):', dnsmos_dfn)

    fila_dfn = {'Version': 'DeepFilterNet (IA)', 'OVRL': dnsmos_dfn.get('ovrl_mos'), 'SIG': dnsmos_dfn.get('sig_mos'), 'BAK': dnsmos_dfn.get('bak_mos')}
    resumen = pd.concat([resumen, pd.DataFrame([fila_dfn])], ignore_index=True)
    display(resumen)
else:
    print(f'No se ha encontrado salida previa de DeepFilterNet en: {RUTA_DFN}')
    print('Ejecuta antes 01_DeepFilterNet_3.ipynb con este mismo audio si quieres esta comparativa.')


ValueError: np.ndarray values must be between -1 and 1.

## 12. Liberar memoria GPU

In [ ]:
liberar_memoria_gpu(modelo_mpsenet, 'modelo_mpsenet')


Memoria GPU liberada. Uso actual: 0.01 GB


## Próximo paso

Con MP-SENet ya probado y comparado contra su baseline (y opcionalmente contra DeepFilterNet), el
siguiente notebook sería AudioSR o VoiceFixer v2, según el orden que fijemos, siguiendo la Fase 2
del proyecto.
